# 02 — Robustification: Is It Worth It?

The nominal solution maximises expected profit under no disruption.  
The robust solution is more conservative — it guards against a worst-case
disruption budget Γ.  

This notebook asks: **does robustification actually help in practice?**

We compare both solutions on a large random sample of disruption scenarios
using five risk-oriented metrics:

| Metric | Interpretation |
|---|---|
| `min_profit` | Absolute worst-case realised profit |
| `pct5_profit` | 5th-percentile profit (VaR at 95 % confidence) |
| `cvar5` / `cvar10` | Mean profit in worst 5 % / 10 % of scenarios (CVaR) |
| `prob_loss` | Fraction of scenarios with negative profit |
| `mean_regret` / `max_regret` | Regret vs. best-case scenario for the same solution |

**One-time setup** — run from the terminal before opening this notebook:
```bash
pip install -e .
```

## 1. Imports and settings

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from rcflp.instance         import instancemaker
from rcflp.nominal          import solve_nominal
from rcflp.ccg              import solve_CCG
from rcflp.scenario_sampler import sample_scenarios
from rcflp.evaluation       import evaluate_solution

In [ ]:
# ── Problem parameters ────────────────────────────────────────────────────────
In  = 15          # customers
Jn  = 10          # candidate facilities
Rn  = 3           # capacity levels
v   = 0.75        # value_max scale
w   = 100         # congestion cost

Hn     = 2        # disruption levels  (0 = intact, 1 = fully disrupted)
gamma  = 3        # uncertainty budget Γ

# ── Robust solver settings ────────────────────────────────────────────────────
OBJ_TOL    = 0.01
BIG_M      = 10_000
TIME_LIMIT = 900   # seconds

# ── Scenario evaluation settings ─────────────────────────────────────────────
N_SCENARIOS = 500
SEED        = 42

## 2. Solve nominal and robust problems

In [ ]:
inst  = instancemaker(In, Jn, Rn, v, w)
nom   = solve_nominal(inst)
x_nom = nom["x_jr"]
print(f"Nominal profit  : {nom['profit']:,.1f}")

rob = solve_CCG(
    inst, gamma, Hn, x_nom,
    tol               = OBJ_TOL,
    big_M             = BIG_M,
    time_limit        = TIME_LIMIT,
    master_mip_gap    = 0.015,
    master_time_limit = 60,
    n_scenarios       = 2,
    L_init            = -nom["profit"],
    eps_e             = 0.03,
    alpha             = 0.8,
    beta              = 120,
    verbose           = False,
)
x_rob = rob["x_jr"]   # {(j,r): float}

print(f"Robust profit LB: {rob['profit_LB']:,.1f}  "
      f"(converged={rob['converged']}, iters={rob['n_iter']})")
print(f"Price of robustness: "
      f"{100*(nom['profit'] - rob['profit_LB'])/nom['profit']:.1f} %")

## 3. Sample disruption scenarios

In [ ]:
scenarios = sample_scenarios(inst, gamma, Hn, n_samples=N_SCENARIOS, seed=SEED)
print(f"Sampled {len(scenarios)} scenarios  (Hn={Hn}, Γ={gamma})")

# Quick budget distribution check
H = list(range(Hn))
budgets = [
    sum((h / (Hn - 1)) * eps[(j, h)] for j in inst["J"] for h in H)
    for eps in scenarios
]
print(f"Budget used — min: {min(budgets):.2f}  "
      f"mean: {np.mean(budgets):.2f}  "
      f"max: {max(budgets):.2f}  (cap={gamma})")

## 4. Evaluate both solutions across scenarios

For each scenario we fix the first-stage decision and solve the second-stage
recourse SOCP to get the realised profit.

In [ ]:
print("Evaluating nominal solution …", flush=True)
eval_nom = evaluate_solution(x_nom, inst, scenarios, Hn)

print("Evaluating robust solution …", flush=True)
eval_rob = evaluate_solution(x_rob, inst, scenarios, Hn)

print("Done.")

## 5. Risk metrics comparison

In [ ]:
rm_nom = eval_nom["risk_metrics"]
rm_rob = eval_rob["risk_metrics"]

metric_labels = {
    "mean_profit" : "Mean profit",
    "min_profit"  : "Min profit",
    "pct5_profit" : "5th-pct profit (VaR 95%)",
    "cvar5"       : "CVaR 5%  (worst 5%  mean)",
    "cvar10"      : "CVaR 10% (worst 10% mean)",
    "prob_loss"   : "P(profit < 0)",
    "mean_regret" : "Mean regret",
    "max_regret"  : "Max regret",
}

rows = []
for key, label in metric_labels.items():
    nom_val = rm_nom[key]
    rob_val = rm_rob[key]
    if key == "prob_loss":
        delta = rob_val - nom_val          # lower is better
        better = "Robust" if rob_val < nom_val else ("Nominal" if nom_val < rob_val else "Tie")
    elif "regret" in key:
        delta = rob_val - nom_val          # lower regret is better
        better = "Robust" if rob_val < nom_val else ("Nominal" if nom_val < rob_val else "Tie")
    else:
        delta = rob_val - nom_val          # higher profit is better
        better = "Robust" if rob_val > nom_val else ("Nominal" if nom_val > rob_val else "Tie")
    rows.append({
        "Metric"       : label,
        "Nominal"      : nom_val,
        "Robust"       : rob_val,
        "Δ (Rob−Nom)"  : delta,
        "Better"       : better,
    })

df_metrics = pd.DataFrame(rows)

def _fmt(val, metric):
    if "prob_loss" in metric.lower() or "p(" in metric.lower():
        return f"{val:.1%}"
    return f"{val:,.1f}"

# Format and colour by winner
df_display = df_metrics.copy()
for col in ["Nominal", "Robust", "Δ (Rob−Nom)"]:
    df_display[col] = [
        _fmt(v, df_metrics["Metric"].iloc[i])
        for i, v in enumerate(df_metrics[col])
    ]

def highlight_winner(row):
    styles = [''] * len(row)
    cols   = list(row.index)
    if row["Better"] == "Robust":
        styles[cols.index("Robust")]  = "background-color: #d4edda"
        styles[cols.index("Nominal")] = "background-color: #f8d7da"
    elif row["Better"] == "Nominal":
        styles[cols.index("Nominal")] = "background-color: #d4edda"
        styles[cols.index("Robust")]  = "background-color: #f8d7da"
    return styles

display(
    df_display.style
    .apply(highlight_winner, axis=1)
    .set_caption(f"Risk metrics — {N_SCENARIOS} sampled scenarios  |  Γ={gamma}")
)

## 6. Profit distribution

In [ ]:
p_nom = np.array(eval_nom["profits"])
p_rob = np.array(eval_rob["profits"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    f"Profit distribution over {N_SCENARIOS} scenarios  "
    f"(|I|={In}, |J|={Jn}, Γ={gamma})",
    fontsize=13,
)

# ── Left: overlaid histograms ─────────────────────────────────────────────────
ax = axes[0]
bins = np.linspace(
    min(p_nom.min(), p_rob.min()),
    max(p_nom.max(), p_rob.max()),
    40,
)
ax.hist(p_nom, bins=bins, alpha=0.55, color="steelblue",  label="Nominal")
ax.hist(p_rob, bins=bins, alpha=0.55, color="darkorange", label="Robust")
ax.axvline(0, color="black", lw=1, ls="--", label="Break-even")
for color, rm in [("steelblue", rm_nom), ("darkorange", rm_rob)]:
    ax.axvline(rm["pct5_profit"], color=color, lw=1.5, ls=":")
ax.set_xlabel("Profit")
ax.set_ylabel("Count")
ax.set_title("Histogram (dotted = 5th pct)")
ax.legend()
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e3:.0f}k"))
ax.grid(True, alpha=0.3)

# ── Right: empirical CDF ──────────────────────────────────────────────────────
ax = axes[1]
for profits, color, label in [
    (p_nom, "steelblue",  "Nominal"),
    (p_rob, "darkorange", "Robust"),
]:
    xs = np.sort(profits)
    ys = np.arange(1, len(xs) + 1) / len(xs)
    ax.plot(xs, ys, color=color, lw=2, label=label)
ax.axvline(0,    color="black", lw=1,   ls="--", label="Break-even")
ax.axhline(0.05, color="grey",  lw=0.8, ls=":")
ax.axhline(0.10, color="grey",  lw=0.8, ls=":")
ax.set_xlabel("Profit")
ax.set_ylabel("Cumulative probability")
ax.set_title("Empirical CDF")
ax.legend()
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e3:.0f}k"))
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Scenario-by-scenario profit gap

Positive values (green) = robust beats nominal in that scenario.

In [ ]:
gap = p_rob - p_nom
order = np.argsort(gap)

fig, ax = plt.subplots(figsize=(14, 4))
colors = ["#2ca02c" if g >= 0 else "#d62728" for g in gap[order]]
ax.bar(range(len(gap)), gap[order], color=colors, width=1.0)
ax.axhline(0, color="black", lw=0.8)
ax.set_xlabel("Scenario (sorted by gap)")
ax.set_ylabel("Robust profit − Nominal profit")
ax.set_title(
    f"Robust advantage per scenario  "
    f"(green={np.sum(gap>=0)}, red={np.sum(gap<0)}, "
    f"mean gap={gap.mean():+,.0f})"
)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e3:.0f}k"))
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Risk metrics bar chart

In [ ]:
profit_keys  = ["mean_profit", "min_profit", "pct5_profit", "cvar5", "cvar10"]
regret_keys  = ["mean_regret", "max_regret"]
prob_keys    = ["prob_loss"]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Risk metric comparison: Nominal vs Robust", fontsize=13)

def _bar_group(ax, keys, labels, title, higher_is_better=True):
    x   = np.arange(len(keys))
    w   = 0.35
    n_v = [rm_nom[k] for k in keys]
    r_v = [rm_rob[k] for k in keys]
    ax.bar(x - w/2, n_v, w, label="Nominal", color="steelblue",  alpha=0.85)
    ax.bar(x + w/2, r_v, w, label="Robust",  color="darkorange", alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.grid(True, axis="y", alpha=0.3)

_bar_group(
    axes[0], profit_keys,
    ["Mean", "Min", "VaR\n95%", "CVaR\n5%", "CVaR\n10%"],
    "Profit metrics (higher = better)",
)
axes[0].yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f"{x/1e3:.0f}k")
)

_bar_group(
    axes[1], regret_keys,
    ["Mean\nregret", "Max\nregret"],
    "Regret vs best-case (lower = better)",
    higher_is_better=False,
)
axes[1].yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f"{x/1e3:.0f}k")
)

_bar_group(
    axes[2], prob_keys,
    ["P(profit<0)"],
    "Loss probability (lower = better)",
    higher_is_better=False,
)
axes[2].yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f"{x:.1%}")
)

plt.tight_layout()
plt.show()

## 9. Summary

The cell below prints a plain-text verdict for each metric.

In [ ]:
print(f"{'Metric':<30}  {'Nominal':>12}  {'Robust':>12}  {'Winner':>8}")
print("-" * 70)
for row in rows:
    m = row["Metric"]
    if "P(" in m:
        n_str = f"{rm_nom['prob_loss']:.1%}"
        r_str = f"{rm_rob['prob_loss']:.1%}"
    else:
        key   = [k for k, v in metric_labels.items() if v == m][0]
        n_str = f"{rm_nom[key]:>12,.1f}"
        r_str = f"{rm_rob[key]:>12,.1f}"
    print(f"{m:<30}  {n_str:>12}  {r_str:>12}  {row['Better']:>8}")

print()
rob_wins = sum(1 for r in rows if r["Better"] == "Robust")
nom_wins = sum(1 for r in rows if r["Better"] == "Nominal")
print(f"Robust wins: {rob_wins}/{len(rows)}   Nominal wins: {nom_wins}/{len(rows)}")
print(f"Price of robustness: "
      f"{100*(nom['profit'] - rob['profit_LB'])/nom['profit']:.1f} % "
      f"of nominal profit")